CODE MARINE + PROGRAMME NATHAN + DIFF TECHNQUE + CHANGEMENT ECHELLE

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg



#------------------------Initialisation----------------------------------------------#
global name_tech
name_tech = "Kyte-Doolittle"
ECHELLE_HYDROPHOBE = {
    "Kyte-Doolittle": {
        "ALA":1.8,"ARG":-4.5,"ASN":-3.5,"ASP":-3.5,"CYS":2.5,
        "GLN":-3.5,"GLU":-3.5,"GLY":-0.4,"HIS":-3.2,"ILE":4.5,
        "LEU":3.8,"LYS":-3.9,"MET":1.9,"PHE":2.8,"PRO":-1.6,
        "SER":-0.8,"THR":-0.7,"TRP":-0.9,"TYR":-1.3,"VAL":4.2
    },
        "Hood-Wood": {
        "ALA":1,"ARG":1,"ASN":1,"ASP":1,"CYS":1,
        "GLN":1,"GLU":1,"GLY":1,"HIS":1,"ILE":1,
        "LEU":3.8,"LYS":-3.9,"MET":1.9,"PHE":2.8,"PRO":-1.6,
        "SER":-0.8,"THR":-0.7,"TRP":-0.9,"TYR":-1.3,"VAL":4.2
    }
}


#-------------------------------LECTURE_PDB--------------------------------------#



def lire_sequence_pdb(nom_f):
    indice = ECHELLE_HYDROPHOBE[name_tech]
    seq = []
    g = open(nom_f, "r")
    ligne = g.readline()
    while ligne != "":
        if ligne[0:6] == "SEQRES":
            long_seq = ligne[19:]
            i = 0
            while i < len(long_seq):
                aa = long_seq[i:i+3]
                i += 4
                if aa in indice:
                    seq += [aa]
        ligne = g.readline()
    g.close()
    print(seq)
    return(seq)

#---------------------------Programme-------------------------------------------#

def moyenne_hydrophobe(seq, fenetre):
    '''Prend un taille de fenetre, un dictionnaire contenant les acides aminés et leur indice d'hydrophobicité ainsi qu'une seq d'aa pour:
    -Pour chaque aa: regardé autour de celui ci (en fonction de la taille de la fenêtre) et faire la moyenne des indices des aa autour
    -Créer une liste de coordonées x,y qu'il renverra
    -profite pour compter les aa et rajouter un 0 à chaque fois pour tracer à la fin la ligne de délimitation'''
    y= [] #listes des coordonées
    indice = ECHELLE_HYDROPHOBE[name_tech]
    demi_f = (fenetre-1)//2
    position_central = demi_f #un coté de la fenetre (gauche)
    while position_central < len(seq)-demi_f: # coté droit, Pour chaque acide aminé
        i = 0
        moyenne = 0
        while i < fenetre and position_central-demi_f+i < len(seq) : 
            #print(position_central, demi_f, i, len(seq))
            if seq[position_central-demi_f+i] not in indice: #Vérifie que l'aa existe
                i +=1
            else:
                moyenne += indice.get(seq[position_central-demi_f+i]) #va additionner les aa à droite et à gauche de notre aa principal (si fenentre = 9, on aura 4aa à droite et 4 aa à gauche)
                #print(seq[k-((fenetre-1)//2)+i], ":", indice.get(seq[k-((fenetre-1)//2)+i]))
                i +=1
        moyenne = moyenne / fenetre #Calcul la moyenne des aa

        #création d'une liste comprenant toutes les coordonées des aa avec leur moyenne d'indice d'hydrophobicité
        y += [moyenne]
        #print("moyenne:", moyenne,"k:", seq[k])
        position_central += 1
    
    return y

# -------- GRAPHIQUE --------
def afficher_profil_hydrophobicite(profil):
    zone_graphe.clear()

    positions_seq = list(range(1, len(profil) + 1))
    valeurs = np.array(profil)

    zone_graphe.plot(positions_seq, valeurs)
    zone_graphe.axhline(0, linestyle="--")

    zone_graphe.fill_between(positions_seq, valeurs, 0, where=(valeurs >= 0), color  = "papayawhip",alpha=1)
    zone_graphe.fill_between(positions_seq, valeurs, 0, where=(valeurs < 0),color = "blue", alpha=0.5)
    

    #Adapte l'echelle de la fenetre
    if len(profil)<120:
        zone_graphe.set_xticks(range(0, len(profil) + 1, 10))
    elif len(profil) <250:
        zone_graphe.set_xticks(range(0, len(profil) + 1, 20))
    elif len(profil) <400 :
        zone_graphe.set_xticks(range(0, len(profil) + 1, 50))
    elif len(profil)<1000 :
        zone_graphe.set_xticks(range(0, len(profil) + 1, 100))
    elif len(profil)< 1500:
        zone_graphe.set_xticks(range(0, len(profil) + 1, 200))
    elif len(profil)<2000:
        zone_graphe.set_xticks(range(0, len(profil) + 1, 400))
        

    zone_graphe.set_ylim(-5, 5)

    zone_graphe.set_title("Profil d'hydrophobicité")
    zone_graphe.set_xlabel("Position dans la séquence")
    zone_graphe.set_ylabel("Indice d'hydrophobicité")

    canvas_graphique.draw()


# -------- ACTION BOUTON --------

def charger_fichier_pdb():
    chemin_fichier = filedialog.askopenfilename(title="Choisir un fichier PDB",filetypes=[("Fichiers PDB", "*.pdb")])
    taille_fenetre=int(fenetre_var.get())
    if not chemin_fichier:
        return

    sequence = lire_sequence_pdb(chemin_fichier)

    if len(sequence) == 0:
        messagebox.showerror("Erreur", "Aucune séquence trouvée")
        return

    label_information.config(text="Longueur : " + str(len(sequence)))

    profil = moyenne_hydrophobe(sequence, taille_fenetre)
    afficher_profil_hydrophobicite(profil)





# -------- INTERFACE --------

fenetre_principale = tk.Tk()
fenetre_principale.geometry("700x600")
fenetre_principale.title("Profil d'hydrophobicité")
fenetre_principale.configure(bg = "maroon")

# grille
i = 0
while i < 20:
    fenetre_principale.grid_rowconfigure(i, weight=1)
    fenetre_principale.grid_columnconfigure(i, weight=1)
    i = i + 1

#bouton
bouton_chargement = tk.Button(
    fenetre_principale,
    text="Choisir fichier PDB",
    command=charger_fichier_pdb,
    bg = "light grey",
    fg = "blue"
)
bouton_chargement.grid(row=0, column=10, columnspan=3, sticky="nsew")

#Label
label_information = tk.Label(fenetre_principale, text="Longueur : ", borderwidth=5, relief = "solid",bg="light grey",fg="red")
label_information.grid(row=1, column=2, columnspan=3)

#graphe
figure_graphique, zone_graphe = plt.subplots(figsize=(4, 3))

canvas_graphique = FigureCanvasTkAgg(figure_graphique, master=fenetre_principale)
canvas_graphique.get_tk_widget().grid(row=10,column=5,columnspan=8,rowspan=8,sticky="snew")

#Input pour fenetre
fenetre_var=tk.StringVar()
fenetre_entry = tk.Entry(fenetre_principale,textvariable = fenetre_var)
fenetre_entry.grid(row=0,column=1)
#Label pour fenetre
label_fenetre = tk.Label(fenetre_principale, text="Taille de fenetre : ",bg="red")
label_fenetre.grid(row=0, column=0)


# Création d'un menu défilant
def changement_k():
    global name_tech
    name_tech = "Kyte-Doolittle"
    print(name_tech)
def changement_h():
    global name_tech
    name_tech = "Hood-Wood"
    print(name_tech)

menuBar = tk.Menu(fenetre_principale)
fenetre_principale.config(menu=menuBar)

menuAffichage = tk.Menu(menuBar)

menuAffichage.add_command(label='K&D', command=changement_k)
menuAffichage.add_command(label="Hood-Wood", command=changement_h)
menuAffichage.add_command(label="Einseberg")

menuBar.add_cascade(label="Technique", menu=menuAffichage)

fenetre_principale.mainloop()